In [1]:
import os
from datetime import datetime
import sys
from pathlib import Path
import shutil
import zipfile
import rarfile
import tarfile
import py7zr
import pandas as pd
from datetime import datetime
from thefuzz import process
import tqdm
from tqdm.auto import tqdm

In [2]:
def new_dir_name(prefix=''):
    now = datetime.now()
    directory = f'{prefix}_{now.strftime("%Y%m%d_%H%M%S")}_{now.microsecond}'
    return directory

In [3]:
def dir_size(dir_path):
    size = 0
    for file in dir_path.rglob('*'):
        if file.is_file():
            size += file.stat().st_size
    return size

In [4]:
def move_file_to_dir(file, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    file_name = file.name
    dest_path = dest_dir / file_name
    shutil.move(file, dest_path)

In [5]:
def contains_file(dir_path):
    for item in dir_path.iterdir():
        if item.is_file():
            return True
    return False

In [6]:
def contains_dir(dir_path):
    for item in dir_path.iterdir():
        if item.is_dir():
            return True
    return False

In [7]:
def manage_zip(zip_file, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
               processed_arch_dir, base_dir, extract_dir):
    #extract zip_file to 00_new_folder
    try:
        new_dir = extract_dir / new_dir_name('mz')
        os.makedirs(new_dir, exist_ok=True)
        with zipfile.ZipFile(zip_file, 'r') as zf:
            zf.extractall(new_dir)
        move_file_to_dir(zip_file, processed_arch_dir)
        #manage(curr_dir, new_dir, irrelevant_files_dir, unprocessed_arch_dir, processed_arch_dir, base_dir, extract_dir)
        #shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(zip_file, unprocessed_arch_dir)

In [8]:
def manage_rar(rar_file, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
               processed_arch_dir, base_dir, extract_dir):
    try:
        new_dir = extract_dir / new_dir_name('mr')
        os.makedirs(new_dir, exist_ok=True)
        with rarfile.RarFile(item) as rf:
            rf.extractall(new_dir)
        move_file_to_dir(rar_file, processed_arch_dir)
        #manage(curr_dir, new_dir, irrelevant_files_dir, unprocessed_arch_dir, processed_arch_dir, base_dir, extract_dir)
        #shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(rar_file, unprocessed_arch_dir)

In [9]:
def manage_tar(tar_file, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
               processed_arch_dir, base_dir, extract_dir):
    try:
        new_dir = extract_dir / new_dir_name('mt')
        os.makedirs(new_dir, exist_ok=True)
        with tarfile.open(tar_file, 'r') as tar:
            tar.extractall(new_dir)
        move_file_to_dir(tar_file, processed_arch_dir)
        #manage(curr_dir, new_dir, irrelevant_files_dir, unprocessed_arch_dir, processed_arch_dir, base_dir, extract_dir)
        #shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(tar_file, unprocessed_arch_dir)

In [10]:
def manage_7z(seven_zip_file, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
              processed_arch_dir, base_dir, extract_dir):
    try:
        new_dir = extract_dir / new_dir_name('m7')
        os.makedirs(new_dir, exist_ok=True)
        with py7zr.SevenZipFile(seven_zip_file, mode='r') as seven_zip:
            seven_zip.extractall(path=new_dir)
        move_file_to_dir(seven_zip_file, processed_arch_dir)
        #manage(curr_dir, new_dir, irrelevant_files_dir, unprocessed_arch_dir, processed_arch_dir, base_dir, extract_dir)
        #shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(seven_zip_file, unprocessed_arch_dir)

In [11]:
def manage_csv(csv_file, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
               processed_arch_dir, base_dir, extract_dir):
    csv_parent = csv_file.parent
    if "_" in csv_file.stem:
        # handle csv files with name prefix
        common_pattern = csv_file.stem.split("_")[0]
        csv_files = [f for f in csv_parent.glob("*.csv") if common_pattern in f.stem]
    else:
        # handle csv files without name prefix
        csv_files = [f for f in csv_parent.glob("*.csv")]

    csv_dest = base_dir / new_dir_name('mc')    
    for csv_file in csv_files:
        move_file_to_dir(csv_file, csv_dest)

In [12]:
def manage(curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
           processed_arch_dir, base_dir, extract_dir):
    num_of_items = len(list(parent_dir.iterdir()))
    for item in tqdm(parent_dir.iterdir(), total=num_of_items, desc="Processing Files..."):
        if item.exists() and item.is_file():
            #rename file to reduce the number of characters in path
            if item.parent != extract_dir:
                if item.suffix.lower().strip() != ".csv":
                    item_new_name = f"{new_dir_name('f')}{item.suffix.lower().strip()}"
                    item_new_path = extract_dir / item_new_name
                    shutil.move(item, item_new_path)
                    item = item_new_path
            
            item_extn = item.suffix.lower().strip()
            if item_extn == ".zip":
                manage_zip(item, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
                           processed_arch_dir, base_dir, extract_dir)
            elif item_extn == ".rar":
                manage_rar(item, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
                           processed_arch_dir, base_dir, extract_dir)
            elif item_extn == ".tar":
                manage_tar(item, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
                           processed_arch_dir, base_dir, extract_dir)
            elif item_extn == ".7z":
                manage_7z(item, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
                          processed_arch_dir, base_dir, extract_dir)
            elif item_extn == ".csv":
                manage_csv(item, curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
                           processed_arch_dir, base_dir, extract_dir)
            else:
                #all other irrelevant files
                move_file_to_dir(item, irrelevant_files_dir)

    num_of_items = len(list(parent_dir.iterdir()))
    for item in tqdm(parent_dir.iterdir(), total=num_of_items, desc="Processing Directories..."):
        if item.exists() and item.is_dir():
            if contains_file(item) or contains_dir(item):
                manage(curr_dir, item, irrelevant_files_dir, unprocessed_arch_dir, 
                       processed_arch_dir, base_dir, extract_dir)
    
    shutil.rmtree(parent_dir, ignore_errors=True)

In [13]:
def main():
    curr_dir = Path.cwd()
    parent_dir = curr_dir / "01_parent_directory"
    irrelevant_files_dir = curr_dir / "02_irrelevant_files"
    unprocessed_arch_dir = curr_dir / "03_unprocessed_archives"
    processed_arch_dir = curr_dir / "04_processed_archives"
    base_dir = curr_dir / "05_base_folder"

    os.makedirs(irrelevant_files_dir, exist_ok=True)
    os.makedirs(unprocessed_arch_dir, exist_ok=True)
    os.makedirs(processed_arch_dir, exist_ok=True)
    os.makedirs(base_dir, exist_ok=True)
    
    manage(curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, 
           processed_arch_dir, base_dir, extract_dir=parent_dir)

In [14]:
main()

Processing Files...:   0%|          | 0/299 [00:00<?, ?it/s]

Processing Directories...:   0%|          | 0/217 [00:00<?, ?it/s]

Processing Files...:   0%|          | 0/748 [00:00<?, ?it/s]

Processing Directories...: 0it [00:00, ?it/s]

Processing Files...:   0%|          | 0/37 [00:00<?, ?it/s]

...

Processing Directories...: 0it [00:00, ?it/s]

Processing Files...:   0%|          | 0/31 [00:00<?, ?it/s]

Processing Directories...: 0it [00:00, ?it/s]

Processing Files...:   0%|          | 0/29 [00:00<?, ?it/s]

Processing Directories...: 0it [00:00, ?it/s]